# SÉANCE 4 : NETTOYAGE AUTOMATISÉ DU DEM (CONSIGNE 1)

Ce notebook implémente le pipeline complet de préparation et correction d'un Modèle Numérique de Terrain (MNT / DEM) :
1. **Détection & élimination du fond** : Passage des pixels hors emprise en `NoData`.
2. **Calage altimétrique** : Ramène les valeurs aberrantes négatives au niveau marin ($0\text{ m}$).
3. **Remplissage des puits (*sinks*)** : Algorithme **Priority-Flood** (Barnes / Wang & Liu) pour résoudre les dépressions artificielles.
4. **Reprojection** : Reprojection vers le système métrique cible (`EPSG:26191` - Merchich / Nord Maroc) avec résolution de $30\text{ m}$.
5. **Traçabilité & Contrôle qualité** : Calcul des statistiques comparatives **Avant / Après** et génération d'un rapport Markdown.

In [ ]:
# =============================================================================
# 0. INSTALLATION DES DÉPENDANCES (décommenter si exécuté sous Google Colab)
# =============================================================================
# !pip install rasterio scipy pandas numpy

## 1. Importation des modules nécessaires

In [ ]:
import heapq
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from scipy.ndimage import label, minimum_filter, binary_erosion

## 2. Configuration des chemins et paramètres

In [ ]:
# Recherche automatique du DEM source selon différents chemins possibles
chemins_possibles = [
    Path("/donnees_brutes/dem/DEM_Grand_Casablanca_30m.tif"),
    Path("/content/donnees_brutes/dem/DEM_Grand_Casablanca_30m.tif"),
    Path.cwd() / "donnees_brutes" / "dem" / "DEM_Grand_Casablanca_30m.tif",
    Path.cwd() / "DEM_Grand_Casablanca_30m.tif",
]
DEM_INPUT = next((p for p in chemins_possibles if p.exists()), None)

if DEM_INPUT is None:
    raise FileNotFoundError("DEM_Grand_Casablanca_30m.tif introuvable ! Vérifiez l'emplacement du fichier source.")

base = Path("/content") if Path("/content").exists() else Path.cwd()
OUT_DIR = base / "donnees_nettoyees" / "dem"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEM_OUTPUT = OUT_DIR / "DEM_Grand_Casablanca_EPSG26191_corrige.tif"
CRS_CIBLE = "EPSG:26191"
NODATA = -9999.0

print(f"[*] DEM source  : {DEM_INPUT}")
print(f"[*] Sortie DEM  : {DEM_OUTPUT}")
print(f"[*] CRS cible   : {CRS_CIBLE}")
print(f"[*] NoData      : {NODATA}")

## 3. Définition des fonctions de traitement et d'analyse

In [ ]:
def detecter_puits(data, valide):
    """Pixels strictement plus bas que tous leurs voisins (hors bords)."""
    temp = np.where(valide, data, np.inf)
    fp = np.ones((3, 3), dtype=bool)
    fp[1, 1] = False
    min_v = minimum_filter(temp, footprint=fp, mode="constant", cval=np.inf)
    puits = valide & (data < min_v)
    puits[0, :] = puits[-1, :] = False
    puits[:, 0] = puits[:, -1] = False
    return puits


def remplir_puits(dem, valide):
    """Priority-Flood (Barnes) : même résultat que le remplissage Wang & Liu."""
    h, w = dem.shape
    filled = dem.copy()
    interieur = binary_erosion(valide, structure=np.ones((3, 3)), border_value=0)
    graines = valide & ~interieur
    ferme = np.zeros_like(valide, dtype=bool)
    tas = []
    for r, c in zip(*np.nonzero(graines)):
        heapq.heappush(tas, (float(filled[r, c]), int(r), int(c)))
        ferme[r, c] = True
    voisins = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
    while tas:
        z, r, c = heapq.heappop(tas)
        for dr, dc in voisins:
            nr, nc = r + dr, c + dc
            if 0 <= nr < h and 0 <= nc < w and valide[nr, nc] and not ferme[nr, nc]:
                ferme[nr, nc] = True
                if filled[nr, nc] < z:
                    filled[nr, nc] = z
                heapq.heappush(tas, (float(filled[nr, nc]), nr, nc))
    return filled


def masque_fond(data, nodata):
    """Valeur de fond = la plus fréquente sur le pourtour ; on ne garde que
    les pixels de cette valeur connectés au bord (le vrai 0 m du littoral reste)."""
    bord = np.concatenate([data[0, :], data[-1, :], data[:, 0], data[:, -1]])
    bord = bord[bord != nodata]
    if bord.size == 0:
        return None, np.zeros(data.shape, bool)
    vals, c = np.unique(bord, return_counts=True)
    fond = float(vals[np.argmax(c)])
    lab, _ = label(data == fond)
    ids = np.unique(np.concatenate([lab[0, :], lab[-1, :], lab[:, 0], lab[:, -1]]))
    ids = ids[ids != 0]
    return fond, np.isin(lab, ids)

## 4. Lecture du raster source et état AVANT

In [ ]:
with rasterio.open(DEM_INPUT) as src:
    data = src.read(1).astype("float32")
    src_crs, src_transform = src.crs, src.transform
    src_w, src_h = src.width, src.height
    bounds, res_avant = src.bounds, src.res[0]
    profil = src.profile.copy()
    nodata_src = src.nodata if src.nodata is not None else NODATA

crs_avant = str(src_crs)
valide_brut = (data != nodata_src) & ~np.isnan(data)

# État AVANT (tel que QGIS le voit : le fond compte comme altitude)
av_min = float(data[valide_brut].min())
av_max = float(data[valide_brut].max())
av_pix = int(valide_brut.sum())

print(f"[*] CRS source            : {crs_avant}")
print(f"[*] Dimensions source     : {src_w} x {src_h}")
print(f"[*] Résolution source     : {res_avant:.5f}")
print(f"[*] Plage brute           : [{av_min:.1f} ; {av_max:.1f}] m")
print(f"[*] Pixels valides bruts  : {av_pix}")

## 5. Détection du fond et masquage en NoData

In [ ]:
fond, m_fond = masque_fond(np.where(valide_brut, data, nodata_src), nodata_src)
data[~valide_brut] = NODATA
data[m_fond] = NODATA
valide = data != NODATA
nb_fond = int(m_fond.sum())
print(f"[*] Valeur de fond détectée : {fond} -> {nb_fond} pixels passés en NoData")

nb_puits_av = int(detecter_puits(data, valide).sum())
nb_neg_av = int((valide & (data < 0)).sum())
print(f"[*] Puits détectés (sinks)    : {nb_puits_av}")
print(f"[*] Pixels négatifs (< 0 m)   : {nb_neg_av}")

## 6. Corrections : Calage altimétrique & Remplissage des puits

In [ ]:
# 1. Calage niveau marin : les valeurs négatives passent à 0 m
data[valide & (data < 0)] = 0.0

# 2. Remplissage des puits via Priority-Flood
data_rempli = remplir_puits(data, valide)

nb_repares = int(((data_rempli != data) & valide).sum())
nb_puits_fill = int(detecter_puits(data_rempli, valide).sum())
print(f"[*] Pixels rehaussés par Priority-Flood : {nb_repares}")
print(f"[*] Puits restants après remplissage    : {nb_puits_fill}")

## 7. Reprojection vers EPSG:26191 (30 m) et sauvegarde GeoTIFF

In [ ]:
transform_c, w_c, h_c = calculate_default_transform(
    src_crs, CRS_CIBLE, src_w, src_h, *bounds, resolution=30.0)
profil.update(crs=CRS_CIBLE, transform=transform_c, width=w_c, height=h_c,
              nodata=NODATA, dtype="float32", compress="deflate")

data_rp = np.full((h_c, w_c), NODATA, dtype=np.float32)
reproject(source=data_rempli.astype(np.float32), destination=data_rp,
          src_transform=src_transform, src_crs=src_crs, src_nodata=NODATA,
          dst_transform=transform_c, dst_crs=CRS_CIBLE, dst_nodata=NODATA,
          resampling=Resampling.bilinear)

with rasterio.open(DEM_OUTPUT, "w", **profil) as dst:
    dst.write(data_rp, 1)

print(f"[+] Fichier généré : {DEM_OUTPUT}")

## 8. État APRÈS et Rapport de traçabilité (Avant / Après)

In [ ]:
valide_ap = data_rp != NODATA
ap_min = float(data_rp[valide_ap].min())
ap_max = float(data_rp[valide_ap].max())
ap_pix = int(valide_ap.sum())
nb_puits_ap = int(detecter_puits(data_rp, valide_ap).sum())

rapport = pd.DataFrame([
    {"Indicateur": "CRS", "Avant": crs_avant, "Apres": CRS_CIBLE,
     "Correction": "Reprojection bilinéaire"},
    {"Indicateur": "Résolution", "Avant": f"{res_avant:.5f} (unités CRS)",
     "Apres": "30 m x 30 m", "Correction": "Résolution métrique"},
    {"Indicateur": "Pixels de fond (hors territoire)", "Avant": f"{nb_fond} comptés comme altitude",
     "Apres": "0 (NoData)", "Correction": f"Valeur {fond} -> NoData {NODATA:.0f}"},
    {"Indicateur": "Pixels valides", "Avant": av_pix, "Apres": ap_pix,
     "Correction": "Territoire réel uniquement"},
    {"Indicateur": "Pixels négatifs", "Avant": nb_neg_av, "Apres": 0,
     "Correction": "Calage à 0 m"},
    {"Indicateur": "Puits (sinks)", "Avant": nb_puits_av,
     "Apres": f"{nb_puits_fill} (après remplissage) / {nb_puits_ap} (après reprojection)",
     "Correction": f"Priority-Flood : {nb_repares} pixels relevés"},
    {"Indicateur": "Plage altimétrique", "Avant": f"[{av_min:.1f} ; {av_max:.1f}] m",
     "Apres": f"[{ap_min:.1f} ; {ap_max:.1f}] m", "Correction": "Fond exclu des statistiques"},
])

print("\n" + "=" * 90)
print("  TABLEAU DE TRAÇABILITÉ DU DEM (AVANT / APRÈS)")
print("=" * 90)
try:
    display(rapport)          # Jupyter / Colab
except NameError:
    print(rapport.to_string(index=False))


def vers_markdown(df):
    lignes = ["| " + " | ".join(df.columns) + " |",
              "|" + "|".join(["---"] * len(df.columns)) + "|"]
    for _, l in df.iterrows():
        lignes.append("| " + " | ".join(str(v) for v in l) + " |")
    return "\n".join(lignes)


fichier_md = OUT_DIR / "rapport_nettoyage_dem_avant_apres.md"
fichier_md.write_text(vers_markdown(rapport), encoding="utf-8")
print(f"\nRaster corrigé : {DEM_OUTPUT}")
print(f"Rapport        : {fichier_md}")